# Analisis Feature Matching pada Citra Augmentasi

Notebook ini digunakan untuk demonstrasi dan analisis feature matching pada final project Computer Vision. Tujuannya adalah membandingkan kemampuan tiga metode deteksi fitur, yaitu SIFT, ORB, dan AKAZE, ketika mencocokkan citra original dengan citra hasil augmentasi: Gaussian blur, Gaussian noise, dan JPEG compression.

Feature matching adalah proses mencari korespondensi antara titik-titik penting pada dua citra. Titik penting ini disebut keypoint, sedangkan representasi numeriknya disebut descriptor. Jika descriptor dari dua citra mirip, maka titik tersebut dapat dianggap sebagai kandidat match.

Pada notebook ini, proses matching dilakukan menggunakan BFMatcher. Untuk menyaring hasil matching, digunakan Lowe Ratio Test dengan threshold 0.75. Match hanya dipertahankan jika jarak descriptor terbaik cukup lebih kecil dibandingkan jarak descriptor terbaik kedua.

## 2. Import Library dan Definisi Path

Bagian ini memuat library yang dibutuhkan dan mendefinisikan path utama project. Notebook akan membaca dataset dari folder `dataset/` serta membaca hasil matching dari `outputs/matching/`.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ORIGINAL_DIR = PROJECT_ROOT / "dataset" / "original" / "train" / "real"
AUGMENTATION_DIRS = {
    "gaussian_blur": PROJECT_ROOT / "dataset" / "gaussian_blur" / "train" / "real",
    "gaussian_noise": PROJECT_ROOT / "dataset" / "gaussian_noise" / "train" / "real",
    "jpeg_compression": PROJECT_ROOT / "dataset" / "jpeg_compression" / "train" / "real",
}

MATCHING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "matching"
SUMMARY_PATH = MATCHING_OUTPUT_DIR / "all_matching_summary.csv"
RATIO_THRESHOLD = 0.75

plt.rcParams["figure.figsize"] = (12, 5)

## 3. Load Original dan Augmented Images

Notebook secara otomatis memilih JPG pertama dari folder citra original. Setelah itu, notebook mencari citra augmentasi yang memiliki id yang sama agar perbandingan dilakukan pada pasangan citra yang sesuai.

Grid berikut menampilkan citra original dan tiga versi augmentasinya. Tahap ini penting sebagai pemeriksaan awal sebelum masuk ke proses deteksi fitur.

In [ ]:
def find_first_jpg(folder: Path) -> Path:
    """Mengambil file JPG pertama berdasarkan urutan nama file."""
    jpg_paths = sorted(
        {
            path
            for pattern in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG")
            for path in folder.glob(pattern)
        }
    )
    if not jpg_paths:
        raise FileNotFoundError(f"Tidak ada file JPG di {folder}")
    return jpg_paths[0]


def get_original_image_id(image_path: Path) -> str:
    """Mengambil id citra dari nama file original."""
    suffix = "_original"
    if not image_path.stem.endswith(suffix):
        raise ValueError(f"Nama file original harus diakhiri dengan '{suffix}': {image_path}")
    return image_path.stem[: -len(suffix)]


def find_augmented_image(original_path: Path, augmentation: str) -> Path:
    """Mencari citra augmentasi yang sesuai dengan citra original."""
    image_id = get_original_image_id(original_path)
    augmentation_dir = AUGMENTATION_DIRS[augmentation]
    for extension in (".jpg", ".jpeg", ".JPG", ".JPEG"):
        candidate = augmentation_dir / f"{image_id}_{augmentation}{extension}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"File augmentasi {augmentation} tidak ditemukan untuk {original_path.name}")


def load_bgr_image(image_path: Path):
    """Memuat citra dengan OpenCV dalam format BGR."""
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Gagal memuat citra: {image_path}")
    return image


def bgr_to_rgb(image):
    """Mengubah citra BGR OpenCV menjadi RGB untuk matplotlib."""
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


original_path = find_first_jpg(ORIGINAL_DIR)
augmented_paths = {
    augmentation: find_augmented_image(original_path, augmentation)
    for augmentation in AUGMENTATION_DIRS.keys()
}

original_bgr = load_bgr_image(original_path)
augmented_bgr = {
    augmentation: load_bgr_image(path)
    for augmentation, path in augmented_paths.items()
}
images_bgr = {"original": original_bgr, **augmented_bgr}

print(f"Original: {original_path}")
for augmentation, path in augmented_paths.items():
    print(f"{augmentation}: {path}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, image) in zip(axes, images_bgr.items()):
    ax.imshow(bgr_to_rgb(image))
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Konversi Grayscale

Deteksi fitur umumnya bekerja pada informasi intensitas. Karena itu, semua citra dikonversi ke grayscale sebelum proses deteksi keypoint dan ekstraksi descriptor.

Visualisasi grayscale di bawah membantu menunjukkan bahwa struktur dasar citra masih terlihat, meskipun setiap augmentasi memberikan perubahan yang berbeda.

In [ ]:
images_gray = {
    name: cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    for name, image in images_bgr.items()
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, image) in zip(axes, images_gray.items()):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"{title} grayscale")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Deteksi Keypoints dan Descriptor

Keypoint adalah titik penting pada citra, misalnya sudut, tepi, atau struktur lokal yang relatif mudah dikenali kembali. Descriptor adalah vektor numerik yang mendeskripsikan pola lokal di sekitar keypoint tersebut.

SIFT menggunakan informasi gradien lokal dan biasanya cukup stabil terhadap perubahan skala serta rotasi. ORB menggunakan FAST keypoint dan BRIEF descriptor yang lebih ringan secara komputasi. AKAZE mendeteksi fitur pada nonlinear scale space dan menghasilkan descriptor biner. Karena jenis descriptor berbeda, SIFT akan dicocokkan dengan `NORM_L2`, sedangkan ORB dan AKAZE menggunakan `NORM_HAMMING`.

Pada bagian ini, semua metode dijalankan pada citra original dan citra augmentasi. Jumlah keypoint ditampilkan sebagai indikator awal seberapa banyak struktur lokal yang berhasil dideteksi.

In [ ]:
METHOD_CONFIGS = {
    "SIFT": {"detector": cv2.SIFT_create(), "norm": cv2.NORM_L2},
    "ORB": {"detector": cv2.ORB_create(), "norm": cv2.NORM_HAMMING},
    "AKAZE": {"detector": cv2.AKAZE_create(), "norm": cv2.NORM_HAMMING},
}

features = {}
for method, config in METHOD_CONFIGS.items():
    features[method] = {}
    detector = config["detector"]
    for image_name, gray_image in images_gray.items():
        keypoints, descriptors = detector.detectAndCompute(gray_image, None)
        features[method][image_name] = {
            "keypoints": keypoints,
            "descriptors": descriptors,
        }

keypoint_counts = pd.DataFrame(
    [
        {
            "method": method,
            "image": image_name,
            "keypoints": len(feature_data["keypoints"]),
        }
        for method, method_features in features.items()
        for image_name, feature_data in method_features.items()
    ]
)

keypoint_counts

In [ ]:
def draw_keypoints(image_bgr, keypoints):
    """Menggambar keypoint di atas citra."""
    return cv2.drawKeypoints(
        image_bgr,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )


methods = list(METHOD_CONFIGS.keys())
image_names = list(images_bgr.keys())

fig, axes = plt.subplots(len(methods), len(image_names), figsize=(16, 10))
for row_idx, method in enumerate(methods):
    for col_idx, image_name in enumerate(image_names):
        ax = axes[row_idx, col_idx]
        keypoint_image = draw_keypoints(
            images_bgr[image_name],
            features[method][image_name]["keypoints"],
        )
        ax.imshow(bgr_to_rgb(keypoint_image))
        ax.set_title(f"{method} - {image_name}")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Feature Matching menggunakan BFMatcher

BFMatcher membandingkan descriptor dari citra original dengan descriptor dari citra augmentasi secara brute-force. Artinya, setiap descriptor dari citra original dibandingkan dengan descriptor pada citra target.

Karena SIFT menghasilkan descriptor bertipe float, digunakan `cv2.NORM_L2`. ORB dan AKAZE menghasilkan descriptor biner, sehingga digunakan `cv2.NORM_HAMMING`. Setelah `knnMatch` dengan `k=2`, Lowe Ratio Test threshold 0.75 digunakan untuk mempertahankan match yang lebih jelas dan mengurangi match ambigu.

In [ ]:
def get_good_matches(descriptors_a, descriptors_b, norm):
    """Melakukan knnMatch dan menerapkan Lowe Ratio Test."""
    if descriptors_a is None or descriptors_b is None or len(descriptors_b) < 2:
        return []

    matcher = cv2.BFMatcher(norm)
    knn_matches = matcher.knnMatch(descriptors_a, descriptors_b, k=2)

    good_matches = []
    for match_pair in knn_matches:
        if len(match_pair) < 2:
            continue
        m, n = match_pair
        if m.distance < RATIO_THRESHOLD * n.distance:
            good_matches.append(m)

    return good_matches


matches = {}
computed_rows = []

for method, config in METHOD_CONFIGS.items():
    matches[method] = {}
    original_features = features[method]["original"]

    for augmentation in AUGMENTATION_DIRS.keys():
        augmented_features = features[method][augmentation]
        good_matches = get_good_matches(
            original_features["descriptors"],
            augmented_features["descriptors"],
            config["norm"],
        )
        matches[method][augmentation] = good_matches

        computed_rows.append(
            {
                "method": method,
                "augmentation": augmentation,
                "keypoints_original": len(original_features["keypoints"]),
                "keypoints_augmented": len(augmented_features["keypoints"]),
                "good_matches": len(good_matches),
            }
        )

computed_summary_df = pd.DataFrame(computed_rows)
computed_summary_df

## 7. Visualisasi Matching

Bagian ini menampilkan hasil visualisasi matching yang sudah disimpan di folder `outputs/matching/`. Grid 3x3 digunakan agar setiap baris menunjukkan metode dan setiap kolom menunjukkan jenis augmentasi.

Visualisasi ini membantu melihat kualitas korespondensi secara langsung, bukan hanya dari jumlah good matches.

In [ ]:
def load_matching_visualization(method: str, augmentation: str):
    """Membaca file PNG hasil matching dari outputs/matching."""
    image_path = MATCHING_OUTPUT_DIR / f"{method.lower()}_original_vs_{augmentation}.png"
    if not image_path.exists():
        raise FileNotFoundError(f"File visualisasi matching tidak ditemukan: {image_path}")
    return plt.imread(image_path)


augmentations = list(AUGMENTATION_DIRS.keys())
fig, axes = plt.subplots(len(methods), len(augmentations), figsize=(18, 12))

for row_idx, method in enumerate(methods):
    for col_idx, augmentation in enumerate(augmentations):
        ax = axes[row_idx, col_idx]
        matching_image = load_matching_visualization(method, augmentation)
        ax.imshow(matching_image)
        ax.set_title(f"{method} vs {augmentation}")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Ringkasan Statistik Matching

Bagian ini membaca file `outputs/matching/all_matching_summary.csv` yang berisi hasil matching untuk seluruh kombinasi metode dan augmentasi. Tabel ini menjadi dasar untuk analisis kuantitatif.

In [ ]:
summary_df = pd.read_csv(SUMMARY_PATH)
summary_df

In [ ]:
good_matches_table = summary_df.pivot(
    index="augmentation",
    columns="method",
    values="good_matches",
)

good_matches_table

## 9. Visualisasi Perbandingan

Bar chart berikut membandingkan jumlah good matches untuk setiap augmentasi. Warna berbeda merepresentasikan metode SIFT, ORB, dan AKAZE.

In [ ]:
ax = good_matches_table[["SIFT", "ORB", "AKAZE"]].plot(
    kind="bar",
    figsize=(10, 6),
    width=0.8,
    color=["#4C78A8", "#F58518", "#54A24B"],
)

ax.set_title("Perbandingan Good Matches pada Setiap Augmentasi")
ax.set_xlabel("Augmentasi")
ax.set_ylabel("Good Matches")
ax.legend(title="Metode")
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.tight_layout()
plt.show()

## 10. Analisis Hasil

Berdasarkan hasil aktual, SIFT menghasilkan good matches paling banyak pada semua augmentasi: 26 pada Gaussian blur, 20 pada Gaussian noise, dan 31 pada JPEG compression. Hal ini menunjukkan bahwa, untuk sample ini, SIFT paling robust dibandingkan ORB dan AKAZE. Salah satu penyebabnya adalah descriptor SIFT merepresentasikan pola local gradient secara lebih kaya, sehingga masih mampu menemukan korespondensi ketika citra mengalami distorsi.

JPEG compression menghasilkan jumlah match tertinggi untuk semua metode: SIFT 31, ORB 9, dan AKAZE 5. Ini mengindikasikan bahwa kompresi JPEG pada sample ini masih mempertahankan struktur lokal penting, meskipun terdapat artefak kompresi. Sebaliknya, Gaussian noise lebih merusak descriptor karena noise menambahkan perubahan intensitas acak pada level piksel. Perubahan acak ini dapat mengganggu pola local gradient yang dipakai untuk membentuk descriptor.

ORB dan AKAZE menghasilkan good matches lebih rendah dibandingkan SIFT. Pada sample ini, ORB menghasilkan 4 match pada blur, 4 pada noise, dan 9 pada JPEG compression. AKAZE menghasilkan 2 match pada blur, 3 pada noise, dan 5 pada JPEG compression. Hasil ini tidak berarti ORB dan AKAZE selalu buruk, tetapi menunjukkan bahwa untuk citra dan parameter default yang digunakan, descriptor biner menghasilkan lebih sedikit korespondensi yang lolos Lowe Ratio Test.

Secara umum, hasil ini menunjukkan hubungan antara jenis distorsi dan performa feature matching. Jika distorsi mengubah struktur lokal dan pola gradien secara kuat, jumlah match cenderung turun. Jika struktur utama masih terjaga, seperti pada JPEG compression di sample ini, descriptor masih dapat menemukan korespondensi yang lebih banyak.

## 11. Kesimpulan

Untuk sample yang dianalisis, SIFT merupakan metode yang paling stabil karena menghasilkan jumlah good matches tertinggi pada semua augmentasi. JPEG compression menjadi augmentasi yang paling sedikit merusak matching, sedangkan Gaussian noise lebih mengganggu descriptor karena mengubah intensitas lokal secara acak.

Kesimpulan utama dari demo ini adalah bahwa performa feature matching sangat dipengaruhi oleh perubahan local feature. Descriptor yang mampu mempertahankan informasi local gradient dengan baik akan lebih kuat dalam menghadapi distorsi citra. Namun, hasil ini tetap perlu dibaca sebagai analisis pada satu sample, sehingga evaluasi lebih luas pada banyak citra diperlukan untuk membuat kesimpulan yang lebih umum.